# ComplianceGPT Phase 2: Automated Evaluation

**Objective:** Evaluate the full pipeline (S7 Retrieval + Generator) against the **True Gold Standard**.

1.  **True Gold Source:** Uses the original Gold CSV for grading, not the S7 results.
2.  **Robust Hydration:** Normalizes Control IDs to ensure successful text lookup from CCS.
3.  **Strict Verification:** Grades against the actual expected control, not just what was retrieved.

---

In [1]:
# --- CELL 1: Install Libraries ---
!pip install -q -U bitsandbytes accelerate pandas
!pip install -q -U transformers

print("Installation complete. IF this is a fresh install, please RESTART RUNTIME now (Menu -> Runtime -> Restart Session).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 117.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Installation complete. IF this is a fresh install, please RESTART RUNTIME now (Menu -> Runtime -> Restart Session).


In [2]:
# --- CELL 2: Setup, Imports & Load Model ---
import os
import sys
import json
import torch
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Setup System Paths (Pointing to generator/verifier folders)

MODULE_PATHS = [
    '/content/drive/MyDrive/answerer_v0/generator/',
    '/content/drive/MyDrive/answerer_v0/generator/verifier/'
]
for path in MODULE_PATHS:
    if path not in sys.path:
        sys.path.append(path)
        print(f"Added to path: {path}")

# 3. Import Project Modules
try:
    from generator import ComplianceGenerator
    from verifier import verify_answer
    print(" Project modules imported successfully.")
except ImportError as e:
    print(f" Import Error: {e}. Please check if files exist in Google Drive.")
    raise e

# 4. Load Model (Qwen 2.5-7B)
print("Loading Model... (This may take a minute)")

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # Initialize ComplianceGPT
    compliance_gpt = ComplianceGenerator(model, tokenizer)
    print(" Model Loaded & compliance_gpt Ready!")

except Exception as e:
    print(f" Model Load Error: {e}")
    print("Restart the runtime after installing bitsandbytes needed.")
    raise e

Mounted at /content/drive
Added to path: /content/drive/MyDrive/answerer_v0/generator/
Added to path: /content/drive/MyDrive/answerer_v0/generator/verifier/
 Project modules imported successfully.
Loading Model... (This may take a minute)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

 Model Loaded & compliance_gpt Ready!


In [3]:
# --- CELL 3: Evaluation Engine (Helper Functions) ---

# === 1. Robust Data Loaders ===
def load_ccs_catalog(path):
    """Recursively loads CCS JSONL text (including nested statements)."""
    db = {}
    if not os.path.exists(path):
        print(f"WARNING: CCS file not found: {path}")
        return db

    def extract_text_recursive(parts_list):
        texts = []
        for p in parts_list:
            if 'prose' in p and p['prose']:
                texts.append(p['prose'])
            if 'parts' in p and isinstance(p['parts'], list):
                texts.append(extract_text_recursive(p['parts']))
        return " ".join(texts)

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line)
                full_id = str(item.get('id', '')).lower().strip()
                simple_id = full_id.split(':')[-1] if ':' in full_id else full_id

                parts = item.get('parts', [])
                statement_texts = []
                if parts:
                    statement_parts = [p for p in parts if p.get('name') == 'statement']
                    if statement_parts:
                        statement_texts.append(extract_text_recursive(statement_parts))
                    else:
                        statement_texts.append(extract_text_recursive(parts))

                fallback = item.get('text') or item.get('statement') or ""
                full_text = (" ".join(statement_texts) + " " + fallback).strip()

                if full_id and full_text: db[full_id] = full_text
                if simple_id and full_text: db[simple_id] = full_text
            except json.JSONDecodeError:
                continue
    return db

def load_csv_robust(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    try:
        return pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin-1')

# === 2. The Universal Evaluation Function ===
def run_evaluation_pipeline(config, output_base_dir):
    """
    Runs the full evaluation loop for a specific dataset configuration.
    """
    dataset_name = config["name"]
    version_tag = config["version"]  # e.g., "rev5" or "rev4"

    print(f"\n{'='*40}")
    print(f"STARTING EVALUATION: {dataset_name}")
    print(f"{'='*40}")

    # 1. Load Data
    print(f"Loading {version_tag} Datasets...")
    ccs_db = load_ccs_catalog(config["ccs_path"])
    df_gold = load_csv_robust(config["gold_path"])
    df_s7 = load_csv_robust(config["s7_path"])
    print(f"Loaded: Gold({len(df_gold)}), S7({len(df_s7)}), CCS({len(ccs_db)})")

    # 2. Main Loop
    results = []
    print(f"Evaluating {len(df_gold)} queries...")

    for _, gold_row in tqdm(df_gold.iterrows(), total=len(df_gold), desc=dataset_name):
        question = gold_row['question']

        # A. Retrieve (from S7 cache)
        s7_subset = df_s7[df_s7['question'] == question].head(5)
        retrieved_docs = []
        if not s7_subset.empty:
            for _, row in s7_subset.iterrows():
                raw_id = str(row['control_id']).strip().lower()
                full_text = ccs_db.get(raw_id, "[Text not found in CCS]")
                retrieved_docs.append({
                    "id": f"NIST_SP800-53:{version_tag}:{row['control_id']}",
                    "text": full_text
                })

        # B. Generate
        try:
            # compliance_gpt MUST be defined in global scope (Cell 2)
            contract = compliance_gpt.generate(question, retrieved_docs, org_profile={})
        except NameError:
            print("CRITICAL ERROR: 'compliance_gpt' model not loaded. Run Cell 2 first.")
            return None

        # C. Verify
        verifier_gold = {
            "id": gold_row.get('id') or gold_row.get('ID'),
            "control_id": gold_row.get('control_id') or gold_row.get('Control ID'),
            "odp_required": gold_row.get('odp_required'),
            "gold_source_version": version_tag  # Pass correct version to verifier
        }
        verdict = verify_answer(contract, verifier_gold, org_profile={})

        # D. Record
        results.append({
            "qid": verifier_gold['id'],
            "question": question,
            "gold_control": verifier_gold['control_id'],
            "is_pass": verdict.is_pass,
            "errors": ";".join(verdict.error_tags),
            "precision": verdict.metrics.get("citation_precision", 0),
            "recall": verdict.metrics.get("citation_recall", 0)
        })

    # 3. Save & Report
    df_res = pd.DataFrame(results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    out_file = os.path.join(output_base_dir, f"Eval_{dataset_name}_{timestamp}.csv")
    os.makedirs(output_base_dir, exist_ok=True)
    df_res.to_csv(out_file, index=False)

    print(f"\n REPORT: {dataset_name}")
    print(f"   Pass Rate:      {df_res['is_pass'].mean():.2%}")
    print(f"   Mean Precision: {df_res['precision'].mean():.2%}")
    print(f"   Mean Recall:    {df_res['recall'].mean():.2%}")
    print(f"   Saved to: {out_file}")

    return df_res

In [4]:
# --- CELL 4: Unified Execution (Rev 5 + Rev 4 + Error Bank) ---

import os
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime

# 1. Define Configurations for all 4 test sets
CONFIGS = [
    # --- Main Gold Sets ---
    {
        "name": "NIST_Rev5_Gold",
        "version": "rev5",
        "ccs_path": "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl",
        "gold_path": "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv",
        "s7_path": "/content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_rev5_results.csv",
        "filter": None
    },
    {
        "name": "NIST_Rev4_Gold",
        "version": "rev4",
        "ccs_path": "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl",
        "gold_path": "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv",
        "s7_path": "/content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_rev4_results.csv",
        "filter": None
    },
    # --- Error Bank (Split by Version) ---
    {
        "name": "ErrorBank_Rev5_Subset",
        "version": "rev5",
        "ccs_path": "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl",
        "gold_path": "/content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv",
        "s7_path": "/content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev5_results.csv",
        "filter": "rev5" # Only run rows where version='rev5'
    },
    {
        "name": "ErrorBank_Rev4_Subset",
        "version": "rev4",
        "ccs_path": "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl",
        "gold_path": "/content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv",
        "s7_path": "/content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev4_results.csv",
        "filter": "rev4" # Only run rows where version='rev4'
    }
]

OUTPUT_DIR = "/content/drive/MyDrive/answerer_v0/generator/verifier/"

# 2. Define the Pipeline Function
def run_evaluation_pipeline_v2(config, output_base_dir):
    dataset_name = config["name"]
    version_tag = config["version"]

    print(f"\n{'='*40}\n STARTING: {dataset_name}\n{'='*40}")

    # Load Data (Robustly)
    # Note: load_ccs_catalog and load_csv_robust must be defined in Cell 3
    ccs_db = load_ccs_catalog(config["ccs_path"])
    try:
        df_gold = load_csv_robust(config["gold_path"])
        df_s7 = load_csv_robust(config["s7_path"])
    except FileNotFoundError as e:
        print(f"SKIPPING: Missing input file. {e}")
        return None

    # --- Apply Filter (for Error Bank split) ---
    if config.get("filter"):
        # Normalize column names to lowercase to avoid 'Version' vs 'version' issues
        df_gold.columns = [c.lower() for c in df_gold.columns]

        if 'version' in df_gold.columns:
            original_len = len(df_gold)
            # Normalize content strings
            df_gold['version'] = df_gold['version'].astype(str).str.lower().str.strip()
            # Filter
            df_gold = df_gold[df_gold['version'] == config['filter']].copy()
            print(f"Filtered Gold Set: {original_len} -> {len(df_gold)} rows ({config['filter']} only)")
        else:
            print(f"Warning: 'version' column not found in CSV, cannot filter. Running full set.")

    print(f"Loaded: Gold({len(df_gold)}), CCS({len(ccs_db)})")

    # Main Evaluation Loop
    results = []
    # Using tqdm for progress bar
    for _, gold_row in tqdm(df_gold.iterrows(), total=len(df_gold), desc=dataset_name):
        question = gold_row['question']

        # A. Retrieve Documents (Lookup from S7 results)
        # We match on the exact question string
        s7_subset = df_s7[df_s7['question'] == question].head(5)
        retrieved_docs = []

        if not s7_subset.empty:
            for _, row in s7_subset.iterrows():
                # Normalize ID for lookup
                raw_id = str(row['control_id']).strip().lower()
                # Hydrate text from CCS
                full_text = ccs_db.get(raw_id, "[Text not found in CCS]")

                # Construct document object for Generator
                retrieved_docs.append({
                    "id": f"NIST_SP800-53:{version_tag}:{row['control_id']}",
                    "text": full_text
                })

        # B. Generate Answer
        try:
            # We pass an empty profile to test the 'Traffic Cop' logic
            contract = compliance_gpt.generate(question, retrieved_docs, org_profile={})
        except NameError:
            print("CRITICAL ERROR: 'compliance_gpt' not loaded. Please run Cell 2.")
            return None

        # C. Verify Answer
        # Handle column name variations (control_id vs Control ID)
        gold_cid = gold_row.get('control_id') or gold_row.get('control id')
        gold_odp = gold_row.get('odp_required')

        verifier_gold = {
            "id": gold_row.get('id'),
            "control_id": gold_cid,
            "odp_required": gold_odp,
            "gold_source_version": version_tag
        }

        verdict = verify_answer(contract, verifier_gold, org_profile={})

        # D. Store Results
        results.append({
            "qid": verifier_gold['id'],
            "question": question,
            "is_pass": verdict.is_pass,
            "precision": verdict.metrics.get("citation_precision", 0),
            "recall": verdict.metrics.get("citation_recall", 0),
            "errors": ";".join(verdict.error_tags)
        })

    # Save Results to CSV
    df_res = pd.DataFrame(results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    outfile_name = f"Eval_{dataset_name}_{timestamp}.csv"
    df_res.to_csv(os.path.join(output_base_dir, outfile_name), index=False)

    return df_res

# 3. Execution Block
final_results = {}
print("Starting Comprehensive Evaluation...")

for config in CONFIGS:
    try:
        df_result = run_evaluation_pipeline_v2(config, OUTPUT_DIR)

        # Aggregate metrics if results exist
        if df_result is not None and not df_result.empty:
            final_results[config["name"]] = {
                "Pass Rate": df_result['is_pass'].mean(),
                "Precision": df_result['precision'].mean(),
                "Recall": df_result['recall'].mean()
            }
    except Exception as e:
        print(f"Error running {config['name']}: {e}")

# 4. Final Summary Table
print("\n" + "="*40 + "\n FINAL SUMMARY TABLE\n" + "="*40)
# Using .map to format percentages (modern pandas compatible)
print(pd.DataFrame(final_results).T.map(lambda x: f"{x:.2%}"))
print("="*40)

Starting Comprehensive Evaluation...

 STARTING: NIST_Rev5_Gold
Loaded: Gold(100), CCS(1013)


NIST_Rev5_Gold:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



 STARTING: NIST_Rev4_Gold
Loaded: Gold(36), CCS(826)


NIST_Rev4_Gold:   0%|          | 0/36 [00:00<?, ?it/s]


 STARTING: ErrorBank_Rev5_Subset
Filtered Gold Set: 34 -> 23 rows (rev5 only)
Loaded: Gold(23), CCS(1013)


ErrorBank_Rev5_Subset:   0%|          | 0/23 [00:00<?, ?it/s]


 STARTING: ErrorBank_Rev4_Subset
Filtered Gold Set: 34 -> 11 rows (rev4 only)
Loaded: Gold(11), CCS(826)


ErrorBank_Rev4_Subset:   0%|          | 0/11 [00:00<?, ?it/s]


 FINAL SUMMARY TABLE
                      Pass Rate Precision   Recall
NIST_Rev5_Gold           97.00%    97.00%   97.00%
NIST_Rev4_Gold          100.00%   100.00%  100.00%
ErrorBank_Rev5_Subset   100.00%   100.00%  100.00%
ErrorBank_Rev4_Subset   100.00%   100.00%  100.00%


Extra Sanity Check:

In [5]:
# --- Data Loading Sanity Check ---
import re
import os

# 1. Define path for the check (Rev 5)
rev5_ccs_path = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"

# 2. Load the DB explicitly for this check
# Note: This relies on load_ccs_catalog being defined in Cell 3
print("Loading Rev 5 CCS for inspection...")
ccs_db = load_ccs_catalog(rev5_ccs_path)

# 3. Run the check
debug_id = "ac-2"

if debug_id in ccs_db:
    print(f"--- RAW TEXT for {debug_id} (Preview) ---")
    print(ccs_db[debug_id][:500] + "...")
    print("-" * 30)

    # Test regex
    current_pattern = r"(?:\{+\s*insert:\s*(?:param,\s*)?([^}]+?)\s*\}+|\[assignment:\s*([^\]]+?)\s*\])"
    matches = re.findall(current_pattern, ccs_db[debug_id], re.IGNORECASE)

    print(f"ODP Matches found: {matches}")

    if matches:
        print("SUCCESS: Control text loaded and ODP placeholders detected.")
    else:
        print("WARNING: Control text loaded, but no ODPs found.")
else:
    print(f"ERROR: Control '{debug_id}' not found in CCS DB!")

Loading Rev 5 CCS for inspection...
--- RAW TEXT for ac-2 (Preview) ---
Define and document the types of accounts allowed and specifically prohibited for use within the system; Assign account managers; Require {{ insert: param, ac-02_odp.01 }} for group and role membership; Specify: Authorized users of the system; Group and role membership; and Access authorizations (i.e., privileges) and {{ insert: param, ac-02_odp.02 }} for each account; Require approvals by {{ insert: param, ac-02_odp.03 }} for requests to create accounts; Create, enable, modify, disable, and rem...
------------------------------
ODP Matches found: [('ac-02_odp.01', ''), ('ac-02_odp.02', ''), ('ac-02_odp.03', ''), ('ac-02_odp.04', ''), ('ac-02_odp.05', ''), ('ac-02_odp.06', ''), ('ac-02_odp.07', ''), ('ac-02_odp.08', ''), ('ac-02_odp.09', ''), ('ac-02_odp.10', '')]
SUCCESS: Control text loaded and ODP placeholders detected.


## pinning versions of libraries used in the notebook for reproducibility

Snapshot the full environment

In [6]:
!python --version
!pip freeze > requirements.lock.txt
!cat requirements.lock.txt


Python 3.12.12
absl-py==1.4.0
accelerate==1.12.0
access==1.1.9
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.2
aiosignal==1.4.0
aiosqlite==0.21.0
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.17.2
altair==5.5.0
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.12.0
anywidget==0.9.21
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2025.12.1.0.45.12
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.1.0
Authlib==1.6.5
autograd==1.8.0
babel==2.17.0
backcall==0.2.0
beartype==0.22.8
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.30.0
bigquery-magics==0.10.3
bitsandbytes==0.48.2
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.1.0
blosc2==3.12.2
bokeh==3.7.3
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.2
catalogue==2.0.10
certifi==2025.11.12
cffi==2.0.0
chardet==5.

Record key “load-bearing” versions

In [7]:
import importlib.metadata as md
for p in ["transformers", "accelerate", "bitsandbytes", "pandas", "pyyaml"]:
    try:
        print(p, md.version(p))
    except:
        print(p, "not installed")


transformers 4.57.3
accelerate 1.12.0
bitsandbytes 0.48.2
pandas 2.3.3
pyyaml 6.0.3
